# Paper Figures — reproduce Fig 2, 3, 4 from cached results

This notebook reads only the cache pickles / JSONs written by the previous three notebooks (or shipped pre-computed in `data/`). It should run end-to-end in under a minute on CPU.

In [ ]:
import os, sys, pickle, numpy as np, pandas as pd, matplotlib.pyplot as plt
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path: sys.path.insert(0, REPO_ROOT)
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

## Fig 2 — Phonetic and Semantic Information Distribution

In [ ]:
cache_dir = os.path.join(REPO_ROOT, 'data', 'euclidean_cache')
import glob
codec_caches = {os.path.basename(p)[:-4]: pickle.load(open(p, 'rb'))
                for p in sorted(glob.glob(os.path.join(cache_dir, '*.pkl')))}
if not codec_caches:
    print('no caches in', cache_dir, '— run notebook 01 first')
else:
    n = len(codec_caches)
    fig, axes = plt.subplots(2, n, figsize=(4*n, 6), squeeze=False)
    for i, (name, gd) in enumerate(codec_caches.items()):
        baseline = gd['random'].mean(axis=0)
        for tag in ['synonym', 'homophone', 'random']:
            axes[0, i].plot(gd[tag].mean(axis=0), marker='o', label=tag)
        axes[0, i].set_title(name); axes[0, i].legend()
        for tag in ['synonym', 'homophone']:
            axes[1, i].plot(gd[tag].mean(axis=0) - baseline, marker='o', label=tag)
        axes[1, i].axhline(0, color='gray', lw=0.5)
        axes[1, i].set_title(f'{name} (− random)'); axes[1, i].legend()
    fig.suptitle('Fig 2 — Phonetic and Semantic Information Distribution')
    plt.tight_layout(); plt.show()

## Fig 3 — Articulation Phonetic Correlation (PWCCA)

In [ ]:
CACHE_DIR = os.path.join(REPO_ROOT, 'data', 'output_stats_cache')
import json, glob
from src.analysis import plot_similarity
caches = sorted([p for p in glob.glob(os.path.join(CACHE_DIR, 'cca_*.json')) if 'mimi_separate' not in p])
if caches:
    fig, axes = plt.subplots(1, len(caches), figsize=(4*len(caches), 3.5), squeeze=False)
    for ax, path in zip(axes.flat, caches):
        d = json.load(open(path))
        plot_similarity(d.get('cca_means'), d.get('rsa_means'), d['num_layers'], d['codec_model'], ax=ax)
    fig.suptitle('Fig 3 — Articulation Phonetic Correlation')
    plt.tight_layout(); plt.show()
else:
    print('no caches in', CACHE_DIR, '— run notebook 02 first')

## Fig 4 — MIMI semantic vs accumulated acoustic

In [ ]:
path = os.path.join(CACHE_DIR, 'cca_similarity_across_layers_mimi_separate.json')
if os.path.exists(path):
    d = json.load(open(path))
    fig, ax = plt.subplots(figsize=(5, 3))
    plot_similarity(d.get('cca_means'), d.get('rsa_means'), d['num_layers'], 'mimi (semantic vs acoustic)', ax=ax)
    plt.show()
else:
    print('Set RUN_FIG4=True in notebook 02 to populate this cache.')

## (Optional) CKA table

In [ ]:
ck_path = os.path.join(REPO_ROOT, 'data', 'cka_results.pkl')
if os.path.exists(ck_path):
    with open(ck_path, 'rb') as f:
        r = pickle.load(f)
    for name, d in r.items():
        print(f'{name}: CKA={d["cka"]:.3f}  baseline={d["baseline_mean"]:.3f}  Δ={d["delta"]:.3f}')
else:
    print('Run notebook 03 to populate', ck_path)